In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

/kaggle/input/competitions/concrete-strength-regression/sample_submission.csv
/kaggle/input/competitions/concrete-strength-regression/train.csv
/kaggle/input/competitions/concrete-strength-regression/test.csv


In [2]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import PowerTransformer
from sklearn.model_selection import KFold
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
from xgboost import XGBRegressor
import os
import joblib

# =================================================================
# 1. "ULTRACLEAN" FEATURE ENGINEERING
# =================================================================
def expert_features(df):
    df = df.copy()
    eps = 1e-6
    df['w_b_ratio'] = df['water'] / (df['cement'] + df['slag'] + df['flyash'] + eps)
    df['log_age'] = np.log1p(df['age'])
    df['cement_age'] = df['cement'] * df['log_age']
    df['binder_richness'] = (df['cement'] + df['slag'] + df['flyash']) / (df['fineaggregate'] + df['coarseaggregate'] + eps)
    
    cols_to_drop = ['age', 'Row ID']
    return df.drop(columns=cols_to_drop, errors='ignore')

# Data loading
train_raw = pd.read_csv("/kaggle/input/competitions/concrete-strength-regression/train.csv")
test_raw = pd.read_csv("/kaggle/input/competitions/concrete-strength-regression/test.csv")

Target = 'csMPa'
X = expert_features(train_raw.drop(columns=[Target]))
# Kept as 2D DataFrame [[Target]] so PowerTransformer doesn't raise ValueError
y = train_raw[[Target]]
X_test_final = expert_features(test_raw)

# 1. Create output directory to store folds and scalers
output_dir = "/kaggle/working/folds"
os.makedirs(output_dir, exist_ok=True)

# Retrieve feature and target column names
feature_columns = list(X.columns)
target_column = Target

print("2. Generating and exporting 10 folds (sub-datasets)...")
print("-" * 70)
kf = KFold(n_splits=10, shuffle=True, random_state=42)

# 2. Iterate through folds
for fold, (train_idx, val_idx) in enumerate(kf.split(X, y)):
    
    X_train_f, X_val_f = X.iloc[train_idx], X.iloc[val_idx]
    y_train_f, y_val_f = y.iloc[train_idx], y.iloc[val_idx]

    # =================================================================
    # 2. DOUBLE POWER TRANSFORM (Yeo-Johnson)
    # =================================================================
    pt_x = PowerTransformer(method='yeo-johnson').fit(X_train_f)
    pt_y = PowerTransformer(method='yeo-johnson').fit(y_train_f)

    # Save X and y scalers
    scaler_features_path = os.path.join(output_dir, f"scale_features_{fold}.joblib")
    scaler_target_path = os.path.join(output_dir, f"scale_target_{fold}.joblib")
    
    joblib.dump(pt_x, scaler_features_path)
    joblib.dump(pt_y, scaler_target_path)

    X_scaled = pt_x.transform(X_train_f)
    y_scaled = pt_y.transform(y_train_f)
    X_val_scaled = pt_x.transform(X_val_f)
    y_val_scaled = pt_y.transform(y_val_f)
    X_test_scaled = pt_x.transform(X_test_final)
    
    # Reconstruct Pandas DataFrames (flatten y to ensure a 1D column)
    df_train = pd.DataFrame(X_scaled, columns=feature_columns)
    df_train[target_column] = y_scaled.flatten()
    
    df_val = pd.DataFrame(X_val_scaled, columns=feature_columns)
    df_val[target_column] = y_val_scaled.flatten()

    df_test = pd.DataFrame(X_test_scaled, columns=feature_columns)
    
    # Define save paths
    train_filename = os.path.join(output_dir, f"train_fold_{fold}_scaled.csv")
    val_filename = os.path.join(output_dir, f"val_fold_{fold}_scaled.csv")
    test_filename = os.path.join(output_dir, f"test_fold_{fold}_scaled.csv")
    
    # Export to CSV (without index)
    df_train.to_csv(train_filename, index=False)
    df_val.to_csv(val_filename, index=False)
    df_test.to_csv(test_filename, index=False)
    
    print(f"Fold {fold:02d} exported -> Train: {df_train.shape} | Val: {df_val.shape} | Test: {df_test.shape}")

print("-" * 70)
print(f"Process completed. Files are saved in: {output_dir}")


2. Generating and exporting 10 folds (sub-datasets)...
----------------------------------------------------------------------
Fold 00 exported -> Train: (648, 12) | Val: (73, 12) | Test: (309, 11)
Fold 01 exported -> Train: (649, 12) | Val: (72, 12) | Test: (309, 11)
Fold 02 exported -> Train: (649, 12) | Val: (72, 12) | Test: (309, 11)
Fold 03 exported -> Train: (649, 12) | Val: (72, 12) | Test: (309, 11)
Fold 04 exported -> Train: (649, 12) | Val: (72, 12) | Test: (309, 11)
Fold 05 exported -> Train: (649, 12) | Val: (72, 12) | Test: (309, 11)
Fold 06 exported -> Train: (649, 12) | Val: (72, 12) | Test: (309, 11)
Fold 07 exported -> Train: (649, 12) | Val: (72, 12) | Test: (309, 11)
Fold 08 exported -> Train: (649, 12) | Val: (72, 12) | Test: (309, 11)
Fold 09 exported -> Train: (649, 12) | Val: (72, 12) | Test: (309, 11)
----------------------------------------------------------------------
Process completed. Files are saved in: /kaggle/working/folds


In [3]:
import os
import copy
import joblib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

# PyTorch natif (100% fonctionnel hors-ligne)
import torch
import torch.nn as nn
from torch.utils.data import TensorDataset, DataLoader
from torch.optim.swa_utils import AveragedModel, SWALR

# =================================================================
# 1. ENVIRONNEMENT HORS-LIGNE & REPRODUCTIBILITÉ
# =================================================================
def seed_everything(seed=42):
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
        torch.backends.cudnn.deterministic = True

seed_everything(42)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Mode hors-ligne validé. Device sélectionné : {device}")

param = 'csMPa'
N_FOLDS = 10
BATCH_SIZE = 32
MAX_EPOCHS = 300
SPARSITY_RATE = 0.30  # 30% des synapses neutralisées (Masque binaire comme 01table)

# =================================================================
# 2. LOCALISATION ROBUSTE DES FICHIERS LOCAUX
# =================================================================
def locate_file(possible_paths):
    for path in possible_paths:
        if os.path.exists(path):
            return path
    raise FileNotFoundError(f"Fichier introuvable : {possible_paths}")

def get_fold_paths(fold):
    train_candidates = [
        f"/kaggle/working/folds/train_fold_{fold}_scaled.csv"
    ]
    val_candidates = [
        f"/kaggle/working/folds/val_fold_{fold}_scaled.csv"
    ]
    test_candidates = [
        f"/kaggle/working/folds/test_fold_{fold}_scaled.csv"
    ]
    scaler_candidates = [
        f"/kaggle/working/folds/scale_target_{fold}.joblib"
    ]
    return (
        locate_file(train_candidates),
        locate_file(val_candidates),
        locate_file(test_candidates),
        locate_file(scaler_candidates)
    )

# =================================================================
# 3. COUCHE LINÉAIRE AVEC MASQUAGE DE HADAMARD (COMPATIBLE DEEPCOPY/SWA)
# =================================================================
class MaskedLinear(nn.Module):
    
    #Implémente W * M (Produit d'Hadamard) de façon native en évitant
    #les hooks non-feuilles de PyTorch pour permettre le deepcopy de SWA.
    
    def __init__(self, in_features, out_features, sparsity=0.30):
        super().__init__()
        self.weight = nn.Parameter(torch.empty(out_features, in_features))
        self.bias = nn.Parameter(torch.empty(out_features))
        
        # Initialisation standard Kaiming
        nn.init.kaiming_uniform_(self.weight, a=np.sqrt(5))
        fan_in, _ = nn.init._calculate_fan_in_and_fan_out(self.weight)
        bound = 1 / np.sqrt(fan_in) if fan_in > 0 else 0
        nn.init.uniform_(self.bias, -bound, bound)

        # Création du masque binaire structurel M in {0, 1}
        if sparsity > 0:
            with torch.no_grad():
                k = int(sparsity * out_features * in_features)
                flat_w = self.weight.abs().view(-1)
                threshold = torch.kthvalue(flat_w, k).values
                mask = (self.weight.abs() > threshold).float()
        else:
            mask = torch.ones(out_features, in_features)

        # Enregistrement en tant que buffer persistant (100% compatible deepcopy)
        self.register_buffer('mask', mask)

    def forward(self, x):
        # Exécution du produit synaptique masqué
        return nn.functional.linear(x, self.weight * self.mask, self.bias)

class ResidualBlock(nn.Module):
    """Bloc résiduel avec LayerNorm, SiLU, Dropout et connexions directes"""
    def __init__(self, dim, dropout=0.15, sparsity=0.30):
        super().__init__()
        self.fc1 = MaskedLinear(dim, dim, sparsity=sparsity)
        self.norm1 = nn.LayerNorm(dim)
        self.act1 = nn.SiLU()
        self.drop = nn.Dropout(dropout)
        self.fc2 = MaskedLinear(dim, dim, sparsity=sparsity)
        self.norm2 = nn.LayerNorm(dim)
        self.act2 = nn.SiLU()

    def forward(self, x):
        residual = x
        out = self.drop(self.act1(self.norm1(self.fc1(x))))
        out = self.norm2(self.fc2(out))
        out = self.act2(out + residual)  # Skip Connection
        return out

class SparseTabularResNet(nn.Module):
    def __init__(self, input_dim, hidden_dim=96, num_blocks=3, dropout=0.15, sparsity=0.30):
        super().__init__()
        self.input_layer = nn.Sequential(
            MaskedLinear(input_dim, hidden_dim, sparsity=sparsity),
            nn.LayerNorm(hidden_dim),
            nn.SiLU()
        )
        self.blocks = nn.ModuleList([
            ResidualBlock(hidden_dim, dropout=dropout, sparsity=sparsity) for _ in range(num_blocks)
        ])
        self.head = nn.Linear(hidden_dim, 1)

    def forward(self, x):
        x = self.input_layer(x)
        for block in self.blocks:
            x = block(x)
        return self.head(x)

# =================================================================
# 4. ENTRAÎNEMENT 10-FOLD AVEC SWA ET COSINE ANNEALING
# =================================================================
print("\n" + "=" * 80)
print(f"ENTRAÎNEMENT DU TABULAR RESNET AVEC SPARSITÉ STRUCTURELLE ({int(SPARSITY_RATE*100)}%)")
print("Technologies activées : Skip-Connections + Masque Hadamard + Cosine Annealing + SWA")
print("=" * 80)
print(f"{'FOLD':<7} | {'TRAIN MSE':<10} | {'VAL MSE (MPa²)':<15} | {'VAL R²':<10}")
print("-" * 68)

_, _, sample_test_path, _ = get_fold_paths(0)
sample_test = pd.read_csv(sample_test_path)
test_preds_total = np.zeros(len(sample_test))
oof_preds_total = []
y_true_all_list = []
fold_mses, fold_r2s = [], []

for fold in range(N_FOLDS):
    tr_path, va_path, te_path, sc_path = get_fold_paths(fold)

    train_df = pd.read_csv(tr_path)
    val_df = pd.read_csv(va_path)
    test_df = pd.read_csv(te_path)
    target_scaler = joblib.load(sc_path)

    # Conversion Tenseurs PyTorch
    X_train = torch.tensor(train_df.drop(columns=['csMPa']).to_numpy(), dtype=torch.float32)
    y_train = torch.tensor(train_df['csMPa'].to_numpy(), dtype=torch.float32).unsqueeze(1)

    X_val = torch.tensor(val_df.drop(columns=['csMPa']).to_numpy(), dtype=torch.float32)
    y_val = torch.tensor(val_df['csMPa'].to_numpy(), dtype=torch.float32).unsqueeze(1)

    X_test = torch.tensor(test_df.to_numpy(), dtype=torch.float32)

    train_loader = DataLoader(TensorDataset(X_train, y_train), batch_size=BATCH_SIZE, shuffle=True)

    # Modèle avec Masquage propre
    model = SparseTabularResNet(
        input_dim=X_train.shape[1],
        hidden_dim=96,
        num_blocks=3,
        dropout=0.15,
        sparsity=SPARSITY_RATE
    ).to(device)

    optimizer = torch.optim.AdamW(model.parameters(), lr=2e-3, weight_decay=1e-3)
    scheduler = torch.optim.lr_scheduler.CosineAnnealingWarmRestarts(optimizer, T_0=50, T_mult=2, eta_min=1e-5)

    # Stochastic Weight Averaging (SWA) - Fonctionne maintenant parfaitement !
    swa_model = AveragedModel(model)
    swa_start = 180
    swa_scheduler = SWALR(optimizer, swa_lr=5e-4)

    criterion = nn.MSELoss()
    best_loss = float("inf")
    best_weights = None

    for epoch in range(MAX_EPOCHS):
        model.train()
        for bx, by in train_loader:
            bx, by = bx.to(device), by.to(device)
            optimizer.zero_grad()
            loss = criterion(model(bx), by)
            loss.backward()

            # Gradient clipping pour stabiliser l'apprentissage
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
            optimizer.step()

        if epoch >= swa_start:
            swa_model.update_parameters(model)
            swa_scheduler.step()
        else:
            scheduler.step()

        # Évaluation périodique
        model.eval()
        with torch.no_grad():
            val_loss = criterion(model(X_val.to(device)), y_val.to(device)).item()

        if val_loss < best_loss and epoch < swa_start:
            best_loss = val_loss
            best_weights = copy.deepcopy(model.state_dict())

    # Inférence avec le modèle SWA régularisé
    swa_model.eval()

    with torch.no_grad():
        pred_tr_sc = swa_model(X_train.to(device)).cpu().numpy()
        pred_va_sc = swa_model(X_val.to(device)).cpu().numpy()
        pred_te_sc = swa_model(X_test.to(device)).cpu().numpy()

    # Reconversion vers MPa réels
    p_train_mpa = target_scaler.inverse_transform(pred_tr_sc).flatten()
    p_val_mpa = target_scaler.inverse_transform(pred_va_sc).flatten()
    p_test_mpa = target_scaler.inverse_transform(pred_te_sc).flatten()

    y_train_real = target_scaler.inverse_transform(y_train.numpy()).flatten()
    y_val_real = target_scaler.inverse_transform(y_val.numpy()).flatten()

    mse_tr = mean_squared_error(y_train_real, p_train_mpa)
    mse_v = mean_squared_error(y_val_real, p_val_mpa)
    r2_v = r2_score(y_val_real, p_val_mpa)

    fold_mses.append(mse_v)
    fold_r2s.append(r2_v)

    oof_preds_total.extend(p_val_mpa)
    y_true_all_list.extend(y_val_real)

    test_preds_total += p_test_mpa / N_FOLDS

    print(f"Fold {fold:02d} | {mse_tr:9.2f} | {mse_v:14.4f} | {r2_v:8.4f}")

# =================================================================
# 5. SYNTHÈSE GLOBALE DES RÉSULTATS
# =================================================================
global_mse = mean_squared_error(y_true_all_list, oof_preds_total)
global_rmse = np.sqrt(global_mse)
global_mae = mean_absolute_error(y_true_all_list, oof_preds_total)
global_r2 = r2_score(y_true_all_list, oof_preds_total)

print("\n" + "=" * 65)
print("SYNTHÈSE DU PYTORCH SPARSE RESNET (OOF EN MPa)")
print("=" * 65)
print(f"MSE Global OOF  : {global_mse:.4f} MPa²")
print(f"RMSE Global OOF : {global_rmse:.4f} MPa")
print(f"MAE Global OOF  : {global_mae:.4f} MPa")
print(f"R² Global OOF   : {global_r2:.4f}")
print(f"Moyenne des MSE : {np.mean(fold_mses):.4f} ± {np.std(fold_mses, ddof=1):.4f} MPa²")
print("=" * 65)

# =================================================================
# 6. GÉNÉRATION STRICTE DU FICHIER OFFICIEL 'submission.csv'
# =================================================================
test_candidates = [
    "/kaggle/input/competitions/concrete-strength-regression/test.csv",
    "/kaggle/input/concrete-strength-regression/test.csv",
    "./test.csv"
]
test_orig_path = locate_file(test_candidates)
test_orig = pd.read_csv(test_orig_path)

row_id_col = "Row ID" if "Row ID" in test_orig.columns else test_orig.columns[0]
submission = pd.DataFrame({
    row_id_col: test_orig[row_id_col],
    'csMPa': test_preds_total
})

submission.to_csv("submission.csv", index=False)
print(f"\nFichier 'submission.csv' généré avec succès en mode hors-ligne ({len(submission)} lignes).")
print("Prêt pour la soumission directe sur Kaggle !")

Mode hors-ligne validé. Device sélectionné : cpu

ENTRAÎNEMENT DU TABULAR RESNET AVEC SPARSITÉ STRUCTURELLE (30%)
Technologies activées : Skip-Connections + Masque Hadamard + Cosine Annealing + SWA
FOLD    | TRAIN MSE  | VAL MSE (MPa²)  | VAL R²    
--------------------------------------------------------------------


/usr/local/lib/python3.13/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but PowerTransformer was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.13/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but PowerTransformer was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.13/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but PowerTransformer was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.13/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but PowerTransformer was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.13/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but PowerTransformer was fitted with feature names
  warnings.warn(


Fold 00 |      3.02 |        18.1879 |   0.9201


/usr/local/lib/python3.13/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but PowerTransformer was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.13/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but PowerTransformer was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.13/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but PowerTransformer was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.13/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but PowerTransformer was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.13/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but PowerTransformer was fitted with feature names
  warnings.warn(


Fold 01 |      3.17 |        14.3946 |   0.9555


/usr/local/lib/python3.13/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but PowerTransformer was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.13/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but PowerTransformer was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.13/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but PowerTransformer was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.13/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but PowerTransformer was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.13/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but PowerTransformer was fitted with feature names
  warnings.warn(


... (10 outputs left out)

Fold 07 |      2.21 |        39.4620 |   0.8864


/usr/local/lib/python3.13/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but PowerTransformer was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.13/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but PowerTransformer was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.13/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but PowerTransformer was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.13/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but PowerTransformer was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.13/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but PowerTransformer was fitted with feature names
  warnings.warn(


Fold 08 |      3.00 |        31.2190 |   0.8897
Fold 09 |      3.32 |        12.5939 |   0.9592

SYNTHÈSE DU PYTORCH SPARSE RESNET (OOF EN MPa)
MSE Global OOF  : 20.3290 MPa²
RMSE Global OOF : 4.5088 MPa
MAE Global OOF  : 2.9431 MPa
R² Global OOF   : 0.9312
Moyenne des MSE : 20.3320 ± 8.8408 MPa²

Fichier 'submission.csv' généré avec succès en mode hors-ligne (309 lignes).
Prêt pour la soumission directe sur Kaggle !


/usr/local/lib/python3.13/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but PowerTransformer was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.13/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but PowerTransformer was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.13/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but PowerTransformer was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.13/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but PowerTransformer was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.13/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but PowerTransformer was fitted with feature names
  warnings.warn(
